# A3 Proposal: data evidence (KITTI Depth Completion)
1. Number of paired samples and annotation types
2. Preliminary pair-quality and distribution analysis
3. Split plan and leakage-prevention unit

Needs `MyDrive/KITTI_DC/dataset/{train,val,test}.tar` (from `kitti_prepare_dataset_colab.ipynb`).

In [ ]:
# Load the dataset (copied from Drive to the Colab disk)
import os, subprocess
OUT, DST = '/content/kitti', '/content/drive/MyDrive/KITTI_DC/dataset'
if not all(os.path.isdir(f'{OUT}/{s}') for s in ('train', 'val', 'test')):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(OUT, exist_ok=True)
    for s in ('train', 'val', 'test'):
        subprocess.run(['tar', '-xf', f'{DST}/{s}.tar', '-C', OUT], check=True)

In [ ]:
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

SPLITS, KINDS = ('train', 'val', 'test'), ('rgb', 'lidar', 'gt')
names = {s: sorted(os.listdir(f'{OUT}/{s}/rgb')) for s in SPLITS}
drive_of = lambda n: n[:21]                                   # '2011_09_26_drive_0017'
depth = lambda p: np.asarray(Image.open(p), np.float32) / 256  # metres, 0 = no data

## 1. Number of paired samples and annotation types

One sample = one frame = **RGB + sparse LiDAR + ground-truth depth**, all 1216 × 352 and pixel-aligned.

| Item | Role | Type | Format |
| :--- | :--- | :--- | :--- |
| RGB | input, modality A | camera image | 8-bit PNG, 3 channels |
| Sparse LiDAR | input, modality B | measured depth at laser points | 16-bit PNG, metres = value / 256, 0 = no point |
| GT depth | **annotation** (target) | per-pixel depth, semi-dense | 16-bit PNG, metres = value / 256, 0 = no label |

The annotation is a **dense regression label** (a distance in metres per pixel), not a class.

In [ ]:
count = pd.DataFrame({'pairs': [len(names[s]) for s in SPLITS],
                      'drives': [len({drive_of(n) for n in names[s]}) for s in SPLITS]}, index=SPLITS)
count.loc['total'] = count.sum()
count

## 2. Preliminary pair-quality and distribution analysis

In [ ]:
# Pair quality: every RGB has a LiDAR and GT file with the same name, and all three have the same size
rows = []
for s in SPLITS:
    complete = sum(all(os.path.exists(f'{OUT}/{s}/{k}/{n}') for k in KINDS) for n in names[s])
    bad_size = sum(len({Image.open(f'{OUT}/{s}/{k}/{n}').size for k in KINDS} | {(1216, 352)}) > 1 for n in names[s])
    rows.append([s, len(names[s]), complete, bad_size])
pd.DataFrame(rows, columns=['split', 'pairs', 'complete triplets', 'size mismatches']).set_index('split')

In [ ]:
# Distribution, on 300 random pairs per split
random.seed(0)
rows, gt_values = [], {}
for s in SPLITS:
    lv, gv, d, diff = [], [], [], []
    for n in random.sample(names[s], min(300, len(names[s]))):
        l, g = depth(f'{OUT}/{s}/lidar/{n}'), depth(f'{OUT}/{s}/gt/{n}')
        lv.append((l > 0).mean()); gv.append((g > 0).mean())
        d.append(g[g > 0][::20])
        both = (l > 0) & (g > 0); diff.append(np.abs(l[both] - g[both]))
    d, diff = np.concatenate(d), np.concatenate(diff)
    gt_values[s] = d
    rows.append([s, 100 * np.mean(lv), 100 * np.mean(gv), np.median(d),
                 100 * (d < 20).mean(), 100 * ((d >= 20) & (d < 40)).mean(), 100 * (d >= 40).mean(), 100 * (diff > 1).mean()])
pd.DataFrame(rows, columns=['split', 'LiDAR valid %', 'GT valid %', 'GT median (m)', 'GT 0-20 m %', 'GT 20-40 m %',
                            'GT >40 m %', 'LiDAR vs GT differ >1 m %']).set_index('split').round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
for s in SPLITS:
    ax[0].hist(gt_values[s], bins=np.arange(0, 86, 2), density=True, histtype='step', lw=2, label=s)
ax[0].set_xlabel('GT depth (m)'); ax[0].set_title('GT depth distribution'); ax[0].legend()
x = 0
for s in SPLITS:
    per_drive = pd.Series([drive_of(n) for n in names[s]]).value_counts().values
    ax[1].bar(range(x, x + len(per_drive)), per_drive, label=s); x += len(per_drive) + 2
ax[1].set_xticks([]); ax[1].set_ylabel('pairs'); ax[1].set_title('Pairs per drive (one bar = one drive)'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# One example pair
n = names['test'][0]
rgb, l, g = Image.open(f'{OUT}/test/rgb/{n}'), depth(f'{OUT}/test/lidar/{n}'), depth(f'{OUT}/test/gt/{n}')
fig, ax = plt.subplots(1, 3, figsize=(18, 2.6))
ys, xs = np.nonzero(l)
ax[0].imshow(rgb); ax[0].set_title('RGB')
ax[1].imshow(rgb); ax[1].scatter(xs, ys, c=l[ys, xs], s=0.3, cmap='turbo', vmin=0, vmax=80); ax[1].set_title('sparse LiDAR')
ax[2].imshow(np.ma.masked_where(g <= 0, g), cmap='turbo', vmin=0, vmax=80); ax[2].set_facecolor('k'); ax[2].set_title('GT depth (black = no label)')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 3. Split plan and leakage-prevention unit

**Leakage-prevention unit: the drive** (one continuous recording = one scene sequence).
Frames are recorded at 10 Hz, so neighbouring frames of a drive are almost identical. A drive is therefore never split:
all its frames go to one split.

| Split | Source |
| :--- | :--- |
| Train | KITTI `train` drives, chosen with a fixed seed, stratified by recording date |
| Val | other KITTI `train` drives, disjoint from Train |
| Test | KITTI's official `val_selection_cropped` (13 drives never used for Train / Val) |

In [ ]:
D = {s: {drive_of(n) for n in names[s]} for s in SPLITS}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    print(f'drives shared by {a} and {b}: {len(D[a] & D[b])}')